# Test of S1-ARD processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-766

In [ ]:
# Experimental configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 3,
        "memory_limit": "12GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_s1ard()

In [ ]:
# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcessor

## Read the tasktables

Documentation: https://cpm.pages.eopf.copernicus.eu/eopf-cpm/main/processor-orchestration-guide/tasktables.html#tasktables

<div class="alert alert-block alert-danger">

Note: not implemented for now for S1-ARD.
</div>

In [ ]:
# NOTE: not implemented for now in S1-ARD
# for process in [DprProcessor.S1ARD]:
#     tasktable: dict = dpr_client.get_process(process.value, cluster_info_eopf)
#     print(f"Tasktable for {process.value!r}:")
#     display(JSON(tasktable))
#     # print(json.dumps(tasktable, indent=2))

## Choose the running mode.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

payload_map = {
    "Run all units": "s1-ard/demo_joborder.yaml",
    "Calibration IW": "s1-ard/demo_joborder_calibration_iw.yaml",
    "Calibration SM": "s1-ard/demo_joborder_calibration_sm.yaml",
    "DEM IW": "s1-ard/demo_joborder_dem_iw.yaml",
    "DEM SM": "s1-ard/demo_joborder_dem_sm.yaml",
    "Reference Geometry IW": "s1-ard/demo_joborder_ref_geom_iw.yaml",
    "Reference Geometry SM": "s1-ard/demo_joborder_ref_geom_sm.yaml",
    "Co-Registration IW": "s1-ard/demo_joborder_coregistration_iw.yaml",
    "Co-Registration SM": "s1-ard/demo_joborder_coregistration_sm.yaml",
}

dropdown = widgets.Dropdown(
    options=payload_map,
    description="Job type:",
    style={'description_width': 'initial'},
)

display(dropdown)

## Init environment for the processors.

In [ ]:
# Get the user's choice
payload_subpath = dropdown.value
print(f"Running configuration file: {dropdown.value!r}")
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir="./config"
)
await dpr.init(local_secrets_file="./config/secrets.json")
    
# Same arguments for all tests
dpr_args = {
    "process": DprProcessor.S1ARD,
    "cluster_info": cluster_info_eopf,
    "payload_subpath": payload_subpath,
    "experimental_config": experimental_config,
}

## Run processor

In [ ]:
# Clean the working dir in the s3 bucket ?
clean_working_dir = True

In [ ]:
# Run S1-ARD
if os.getenv("RSPY_FROM_CICD") != "1":
    s3_output_dir = osp.join(dpr.s3_output_dir, "s1ard")
    s3_working_dir = osp.join(dpr.s3_working_dir, "s1ard")
    await dpr.run_process(
        **dpr_args,
        s3_output_dir = s3_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "s1ard"),
        del_s3_working_dir = s3_working_dir if clean_working_dir else "",
        # Payload env vars
        OUTPUT_DIR = s3_output_dir,
        WORKING_DIR = s3_working_dir,
    )